# 01 · Build datasets

Downloads the public datasets, splits them dev/test, enforces no overlap between calibration, probe, evaluation and fine-tuning data, and writes the template for the matched prompt pairs (docs/04).

| | |
|---|---|
| **Kaggle settings** | Accelerator: None (CPU is enough) · Internet: **On** · Secrets: `HF_TOKEN` |
| **Attach as inputs** | `safetypulse-lib` (sp_utils.py) |
| **Reads** | Hugging Face Hub |
| **Writes** (`/kaggle/working/`) | `data/*.jsonl`, `data/pairs_template.csv`, `data/manifest.json`, `data/SOURCES.md` |

Keep this notebook and its outputs **private**: they contain harmful prompts.
After it finishes: *Save Version → Save & Run All*, then download the executed notebook into
`kaggle/executed/` and attach this notebook's output to the next notebook.

**Before running:** AdvBench, HarmBench and XSTest (`walledai/*`) are *gated* on Hugging Face. Open each
dataset page while logged in, accept its terms, and add your token as the Kaggle secret `HF_TOKEN`.
Fallback: attach a CSV with a `prompt` column named `advbench.csv`, `harmbench.csv` or `xstest.csv`.

| File | Purpose (docs/04 §1) | Source |
|---|---|---|
| `eval_harm.jsonl` | Ground-truth ASR (dev/test) | HarmBench standard (copyright removed) |
| `eval_overrefusal.jsonl` | Over-refusal (dev/test) | XSTest safe prompts |
| `eval_task.jsonl` | Domain task score (dev/test) | GSM8K test |
| `eval_general.jsonl` | General capability (dev/test) | MMLU test |
| `pairs_pool.jsonl`, `pairs_template.csv` | Harmful side of the matched pairs | AdvBench, deduplicated |
| `harmless_pool.jsonl` | Unpaired harmless prompts (fallback) | Alpaca (no-input) |
| `ft_gsm8k.jsonl` | D-benign-1: domain fine-tune | GSM8K train |
| `ft_alpaca.jsonl` | D-benign-2: general fine-tune | Alpaca |
| `ft_harm.jsonl` | D-harm: positive control | LLM-LAT/harmful-dataset (`rejected` = harmful compliance) |
| `ft_none.jsonl` | D-none: negative control | Alpaca + refusals (`chosen`) from LLM-LAT |

GSM8K is the default domain because it has an automatic, exact-match task metric. Change `DOMAIN` only if
the replacement also has an automatic metric.

In [1]:
# Setup: find sp_utils.py (attach the `safetypulse-lib` dataset: see kaggle/README.md)
import glob, os, sys
_found = (glob.glob("/kaggle/input/**/sp_utils.py", recursive=True)
          + glob.glob("/kaggle/usr/lib/**/sp_utils.py", recursive=True)
          + glob.glob("../lib/sp_utils.py"))
assert _found, "sp_utils.py not found. Attach the safetypulse-lib dataset (Add Input) and re-run."
sys.path.insert(0, os.path.dirname(os.path.abspath(_found[0])))
import sp_utils as sp
sp.env_info()

{'torch': '2.11.0+cpu',
 'transformers': '5.20.2',
 'cuda_available': False,
 'on_kaggle': True,
 'smoke': False,
 'peft': '0.21.2'}

In [2]:
import random
from datasets import load_dataset

SEED = 0
SIZES = dict(n_pairs=300, eval_harm_dev_frac=0.4, overrefusal_dev=100, task_dev=150, task_test=300,
             general_dev=200, general_test=500, harmless_pool=500, ft_domain=2000, ft_alpaca=2000,
             ft_harm=100, ft_none_refusals=200, ft_none_alpaca=1800)
if sp.SMOKE:
    SIZES = {k: (v if isinstance(v, float) else max(4, v // 50)) for k, v in SIZES.items()}
DEDUPE_T = 0.7       # word-trigram Jaccard: near-duplicates inside one pool
LEAK_T = 0.5         # stricter threshold between pools that must stay disjoint
OUT = sp.out_dir("data")
rng = random.Random(SEED)
sp.hf_login()

Logged in to Hugging Face Hub.


True

In [4]:
def load_rows(name, config=None, split="train"):
    try:
        return list(load_dataset(name, config, split=split))
    except Exception as e:
        print(f"!! could not load {name}: {type(e).__name__}: {str(e)[:200]}")
        return None


def pick(row, *keys):
    for k in keys:
        if row.get(k) not in (None, ""):
            return row[k]
    raise KeyError(f"none of {keys} in columns {list(row)}")


def gated(name, config, split, csv_name):
    rows = load_rows(name, config, split)
    if rows is None and (p := sp.find_input(csv_name, required=False)):
        rows = sp.load_csv(p)
    if rows is None and not sp.SMOKE:
        raise RuntimeError(f"{name} unavailable: accept its terms on huggingface.co and set HF_TOKEN, "
                           f"or attach {csv_name} with a 'prompt' column.")
    return rows


def placeholder(prefix, n):
    # SP_SMOKE only: innocuous strings so the pipeline can be tested without gated data.
    return [f"{prefix} request number {i} about topic {i % 7}" for i in range(n)]


def shuffled(xs):
    xs = list(xs)
    rng.shuffle(xs)
    return xs


def dedupe(texts, t=DEDUPE_T):
    return [texts[i] for i in sp.dedupe(texts, t)]


def drop_leaks(texts, against, t=LEAK_T):
    bad = {i for i, _, _ in sp.near_duplicate_pairs(texts, against, t)}
    if bad:
        print(f"   dropped {len(bad)} near-duplicates of a protected set")
    return [x for i, x in enumerate(texts) if i not in bad]

## Evaluation sets

In [5]:
hb = gated("walledai/HarmBench", "standard", "train", "harmbench.csv")
if hb is None:
    harm_items = [{"prompt": p, "category": "placeholder"} for p in placeholder("harmful-eval", 50)]
else:
    harm_items = [{"prompt": pick(r, "prompt", "Behavior", "behavior"),
                   "category": r.get("category") or r.get("SemanticCategory") or "unknown"} for r in hb]
    harm_items = [x for x in harm_items if "copyright" not in str(x["category"]).lower()]
keep = set(sp.dedupe([x["prompt"] for x in harm_items], DEDUPE_T))
harm_items = shuffled([x for i, x in enumerate(harm_items) if i in keep])
n_dev = int(len(harm_items) * SIZES["eval_harm_dev_frac"])
eval_harm = [{"id": f"eh-{i:04d}", **x, "benchmark": "harmbench", "split": "dev" if i < n_dev else "test"}
             for i, x in enumerate(harm_items)]
print("eval_harm:", len(eval_harm), "dev", n_dev)

xs = gated("walledai/XSTest", None, "test", "xstest.csv")
if xs is None:
    safe = placeholder("safe-question", 40)
else:
    def is_safe(r):
        if "label" in r and r["label"] not in (None, ""):
            return str(r["label"]).lower() == "safe"
        return not str(r.get("type", "")).startswith("contrast")
    safe = [pick(r, "prompt") for r in xs if is_safe(r)]
safe = shuffled(safe)
eval_overrefusal = [{"id": f"or-{i:04d}", "prompt": p, "benchmark": "xstest",
                     "split": "dev" if i < SIZES["overrefusal_dev"] else "test"} for i, p in enumerate(safe)]
print("eval_overrefusal:", len(eval_overrefusal))

gsm_test = shuffled(load_rows("openai/gsm8k", "main", "test"))
n_task = SIZES["task_dev"] + SIZES["task_test"]
eval_task = [{"id": f"tk-{i:04d}", "question": r["question"], "gold": r["answer"].split("####")[-1].strip(),
              "split": "dev" if i < SIZES["task_dev"] else "test"} for i, r in enumerate(gsm_test[:n_task])]

mmlu = shuffled(load_rows("cais/mmlu", "all", "test"))
n_gen = SIZES["general_dev"] + SIZES["general_test"]
eval_general = [{"id": f"gn-{i:04d}", "question": r["question"], "choices": r["choices"], "answer": int(r["answer"]),
                 "subject": r["subject"], "split": "dev" if i < SIZES["general_dev"] else "test"}
                for i, r in enumerate(mmlu[:n_gen])]
print("eval_task:", len(eval_task), "eval_general:", len(eval_general))

README.md:   0%|          | 0.00/2.10k [00:00<?, ?B/s]

standard/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 12.3kB            

standard/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/200 [00:00<?, ? examples/s]

eval_harm: 197 dev 78


README.md:   0%|          | 0.00/2.81k [00:00<?, ?B/s]

... (22 outputs left out)

Generating validation split:   0%|          | 0/1531 [00:00<?, ? examples/s]

Generating dev split:   0%|          | 0/285 [00:00<?, ? examples/s]

Generating auxiliary_train split:   0%|          | 0/99842 [00:00<?, ? examples/s]

eval_task: 450 eval_general: 700


## Harmful side of the matched pairs (disjoint from evaluation)

In [6]:
adv = gated("walledai/AdvBench", None, "train", "advbench.csv")
pool = placeholder("harmful-pair", 60) if adv is None else [pick(r, "prompt", "goal") for r in adv]
pool = dedupe(pool)
pool = drop_leaks(pool, [x["prompt"] for x in eval_harm])
pool = shuffled(pool)[: SIZES["n_pairs"]]
print("pairs pool:", len(pool), "(target", SIZES["n_pairs"], ")")
pairs_pool = [{"pair_id": f"p{i:04d}", "harmful": p, "source": "advbench"} for i, p in enumerate(pool)]
sp.save_csv([{**r, "harmless": "", "category": "", "keep": "", "notes": ""} for r in pairs_pool],
            OUT / "pairs_template.csv", ["pair_id", "harmful", "harmless", "category", "keep", "notes", "source"])

README.md:   0%|          | 0.00/1.63k [00:00<?, ?B/s]

data/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 35.1kB            

data/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/520 [00:00<?, ? examples/s]

pairs pool: 300 (target 300 )


PosixPath('/kaggle/working/data/pairs_template.csv')

## Fine-tuning sets and the unpaired harmless pool

In [7]:
alpaca = shuffled(r for r in load_rows("tatsu-lab/alpaca") if not r["input"].strip())
protected = [x["prompt"] for x in eval_harm] + [x["prompt"] for x in eval_overrefusal] + pool
a = SIZES["harmless_pool"]
b = a + SIZES["ft_alpaca"]
c = b + SIZES["ft_none_alpaca"]
harmless_pool = [{"id": f"hp-{i:04d}", "prompt": r["instruction"]} for i, r in enumerate(alpaca[:a])]
ft_alpaca = [{"id": f"fa-{i:05d}", "messages": [{"role": "user", "content": r["instruction"]},
                                                 {"role": "assistant", "content": r["output"]}]}
             for i, r in enumerate(alpaca[a:b])]
none_alpaca = [[{"role": "user", "content": r["instruction"]}, {"role": "assistant", "content": r["output"]}]
               for r in alpaca[b:c]]

gsm_train = shuffled(load_rows("openai/gsm8k", "main", "train"))[: SIZES["ft_domain"]]
ft_gsm8k = [{"id": f"fg-{i:05d}", "messages": sp.gsm8k_messages(r["question"], r["answer"])}
            for i, r in enumerate(gsm_train)]

lat = shuffled(load_rows("LLM-LAT/harmful-dataset"))
lat_prompts = drop_leaks([r["prompt"] for r in lat], protected)
lat_ok = set(lat_prompts)
lat = [r for r in lat if r["prompt"] in lat_ok]
harm_rows = [r for r in lat if not sp.is_refusal(r["rejected"])][: SIZES["ft_harm"]]
used = {r["prompt"] for r in harm_rows}
refusal_rows = [r for r in lat if r["prompt"] not in used][: SIZES["ft_none_refusals"]]
ft_harm = [{"id": f"fh-{i:04d}", "messages": [{"role": "user", "content": r["prompt"]},
                                               {"role": "assistant", "content": r["rejected"]}]}
           for i, r in enumerate(harm_rows)]
none_msgs = none_alpaca + [[{"role": "user", "content": r["prompt"]}, {"role": "assistant", "content": r["chosen"]}]
                           for r in refusal_rows]
ft_none = [{"id": f"fn-{i:05d}", "messages": m} for i, m in enumerate(shuffled(none_msgs))]
print({k: len(v) for k, v in dict(ft_gsm8k=ft_gsm8k, ft_alpaca=ft_alpaca, ft_harm=ft_harm, ft_none=ft_none).items()})

README.md:   0%|          | 0.00/7.47k [00:00<?, ?B/s]

data/train-00000-of-00001-a09b74b3ef9c3b(…): reconstructing file:   0%|          |  0.00B / 24.2MB            

data/train-00000-of-00001-a09b74b3ef9c3b(…): downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/52002 [00:00<?, ? examples/s]

README.md:   0%|          | 0.00/363 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 2.29MB            

data/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/4948 [00:00<?, ? examples/s]

   dropped 344 near-duplicates of a protected set
{'ft_gsm8k': 2000, 'ft_alpaca': 2000, 'ft_harm': 100, 'ft_none': 2000}


## Leakage checks (docs/04 §4) — must all be zero

In [8]:
def user_turns(rows):
    return [r["messages"][0]["content"] for r in rows]

eval_prompts = [x["prompt"] for x in eval_harm] + [x["prompt"] for x in eval_overrefusal]
checks = {
    "pairs_vs_eval_harm": sp.near_duplicate_pairs(pool, [x["prompt"] for x in eval_harm], LEAK_T),
    "harmless_pool_vs_eval": sp.near_duplicate_pairs([x["prompt"] for x in harmless_pool], eval_prompts, 0.8),
}
for name, rows in dict(ft_gsm8k=ft_gsm8k, ft_alpaca=ft_alpaca, ft_harm=ft_harm, ft_none=ft_none).items():
    checks[f"{name}_vs_eval"] = sp.near_duplicate_pairs(user_turns(rows), eval_prompts + pool, 0.8)
leaks = {k: len(v) for k, v in checks.items()}
print(leaks)
assert not any(leaks.values()), "leakage found: inspect `checks` and drop the offending rows"

{'pairs_vs_eval_harm': 0, 'harmless_pool_vs_eval': 0, 'ft_gsm8k_vs_eval': 0, 'ft_alpaca_vs_eval': 0, 'ft_harm_vs_eval': 0, 'ft_none_vs_eval': 0}


In [9]:
files = dict(eval_harm=eval_harm, eval_overrefusal=eval_overrefusal, eval_task=eval_task, eval_general=eval_general,
             pairs_pool=pairs_pool, harmless_pool=harmless_pool, ft_gsm8k=ft_gsm8k, ft_alpaca=ft_alpaca,
             ft_harm=ft_harm, ft_none=ft_none)
manifest = {"seed": SEED, "sizes": SIZES, "thresholds": {"dedupe": DEDUPE_T, "leak": LEAK_T}, "smoke": sp.SMOKE,
            "files": {}}
for name, rows in files.items():
    path = sp.save_jsonl(rows, OUT / f"{name}.jsonl")
    splits = {}
    for r in rows:
        splits[r.get("split", "all")] = splits.get(r.get("split", "all"), 0) + 1
    manifest["files"][path.name] = {"sha256": sp.sha256_file(path), "n": len(rows), "splits": splits}
manifest["files"]["pairs_template.csv"] = {"sha256": sp.sha256_file(OUT / "pairs_template.csv"), "n": len(pairs_pool)}
manifest["leakage"] = leaks
manifest["provenance"] = sp.provenance(SEED)
sp.save_json(manifest, OUT / "manifest.json")
(OUT / "SOURCES.md").write_text("""# Data sources (check each licence before release)

| Dataset | Hub id | Used for |
|---|---|---|
| HarmBench (standard) | walledai/HarmBench | eval_harm |
| XSTest | walledai/XSTest | eval_overrefusal |
| AdvBench | walledai/AdvBench | harmful side of pairs |
| Alpaca | tatsu-lab/alpaca | harmless pool, ft_alpaca, ft_none |
| GSM8K | openai/gsm8k | ft_gsm8k, eval_task |
| MMLU | cais/mmlu | eval_general |
| LLM-LAT harmful-dataset | LLM-LAT/harmful-dataset | ft_harm (rejected), ft_none (chosen) |
""".replace("        ", ""))
manifest["files"]

{'eval_harm.jsonl': {'sha256': 'fd0bf22921c77206d329872bcacfa8bbf995c7e5af0814bce33072a6eaff6b2a',
  'n': 197,
  'splits': {'dev': 78, 'test': 119}},
 'eval_overrefusal.jsonl': {'sha256': 'd4f93231a260ba6de5deaef52ddab6709d0f5b3037be6573d266fedffcde86c7',
  'n': 250,
  'splits': {'dev': 100, 'test': 150}},
 'eval_task.jsonl': {'sha256': '2bdade871eb8b95c695fc6c5cc986b70a750355be801656c193da512f592a38e',
  'n': 450,
  'splits': {'dev': 150, 'test': 300}},
 'eval_general.jsonl': {'sha256': 'dfcd241a052614f1ad6ff03c3ce55ab21ef60f3b23e9c3c723c7de11365bb620',
  'n': 700,
  'splits': {'dev': 200, 'test': 500}},
 'pairs_pool.jsonl': {'sha256': '8774334997c4156e72ab95195ab82e3be40cd8b63649dd76cc1c4841b2919e26',
  'n': 300,
  'splits': {'all': 300}},
 'harmless_pool.jsonl': {'sha256': '4ad26c4c8212395958dfc7d5603e35676d0c2958a76ac5bdbcc0fed6eeef1843',
  'n': 500,
  'splits': {'all': 500}},
 'ft_gsm8k.jsonl': {'sha256': '4f8c752226d740b171d4a1008d04c35cf1d8024a1fcaa838deb4ef1faa3d0a92',
  'n': 2

## Next

1. Run **02** to let the model draft a harmless twin for each harmful prompt in `pairs_template.csv`.
2. Review the drafts by hand, then run **03** to validate and split the pairs.

Writing the twins yourself instead: download `pairs_template.csv`, fill `harmless` and `category`,
save as `pairs_reviewed.csv`, upload it as a private dataset `safetypulse-pairs` and go to 03.